# VoyageAI - Itinerary Agent

This notebook creates the Itinerary Agent for VoyageAI.

The Itinerary Agent is responsible for:
1. Combining all previous agent outputs
2. Creating a day-wise travel plan
3. Including stay, food, transport, weather, and budget guidance
4. Respecting budget status
5. Returning a structured final itinerary

This agent does not calculate budget itself.
It uses the Budget Agent result.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import json

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

e:\Agentic AI\VoyageAI Backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

print("Environment variables loaded.")

Environment variables loaded.


In [3]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "agents":
    PROJECT_ROOT = CURRENT_DIR.parent.parent
elif CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

CHROMA_DIR = PROJECT_ROOT / "chroma_db"

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Chroma DB folder:", CHROMA_DIR)

if not CHROMA_DIR.exists():
    raise FileNotFoundError("Chroma DB folder not found. Run 01_rag_ingestion.ipynb first.")

Current directory: e:\Agentic AI\VoyageAI Backend\notebooks\agents
Project root: e:\Agentic AI\VoyageAI Backend
Chroma DB folder: e:\Agentic AI\VoyageAI Backend\chroma_db


In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2990.73it/s]


Embedding model loaded successfully.


In [5]:
vector_store = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embedding_model,
    collection_name="voyageai_travel_knowledge"
)

print("ChromaDB loaded successfully.")
print("Total vectors:", vector_store._collection.count())

ChromaDB loaded successfully.
Total vectors: 45


In [6]:
def get_available_destinations():
    collection_data = vector_store._collection.get(include=["metadatas"])

    destinations = set()

    for metadata in collection_data["metadatas"]:
        city = metadata.get("city")
        if city:
            destinations.add(city.lower().strip())

    return sorted(list(destinations))


available_destinations = get_available_destinations()

print("Available destinations in ChromaDB:")
print(available_destinations)

Available destinations in ChromaDB:
['agra', 'andaman', 'darjeeling', 'delhi', 'goa', 'jaipur', 'kashmir', 'kerala', 'ladakh', 'manali', 'mumbai', 'pondicherry', 'rishikesh', 'udaipur', 'varanasi']


In [7]:
def normalize_text(text: str):
    if not text:
        return ""

    return text.lower().strip()


def is_destination_available(destination_name: str):
    destination_name = normalize_text(destination_name)

    if not destination_name:
        return False

    for destination in available_destinations:
        if destination == destination_name:
            return True

        if destination in destination_name:
            return True

        if destination_name in destination:
            return True

    return False

In [9]:
DESTINATION_KEY_MAP = {
    "goa": "goa",
    "jaipur": "jaipur",
    "manali": "manali",
    "kerala": "kerala",
    "andaman": "andaman",
    "andaman and nicobar islands": "andaman",
    "udaipur": "udaipur",
    "rishikesh": "rishikesh",
    "varanasi": "varanasi",
    "ladakh": "ladakh",
    "kashmir": "kashmir",
    "darjeeling": "darjeeling",
    "pondicherry": "pondicherry",
    "puducherry": "pondicherry",
    "agra": "agra",
    "delhi": "delhi",
    "mumbai": "mumbai"
}


def normalize_destination_to_key(destination_name: str):
    if not destination_name:
        return None

    destination_name = destination_name.lower().strip()

    for name, key in DESTINATION_KEY_MAP.items():
        if name in destination_name:
            return key

    return destination_name.replace(" ", "_")

In [10]:
def get_itinerary_context(user_query: str, destination_result: dict, k: int = 5):
    destination = destination_result.get("recommended_destination")
    city_key = normalize_destination_to_key(destination)

    retrieval_query = f"""
    Destination: {destination}
    User request: {user_query}
    Need attractions, food, stay suggestions, best time, travel tips, ideal duration, day-wise itinerary planning.
    """

    try:
        docs = vector_store.similarity_search(
            retrieval_query,
            k=k,
            filter={"city": city_key}
        )

        if not docs:
            docs = vector_store.similarity_search(retrieval_query, k=k)

    except Exception as e:
        print("Filter search failed. Running normal similarity search.")
        print("Error:", e)
        docs = vector_store.similarity_search(retrieval_query, k=k)

    context_parts = []

    for i, doc in enumerate(docs, start=1):
        city = doc.metadata.get("city")
        source = doc.metadata.get("source")

        context = f"""
Context {i}
City: {city}
Source: {source}
Content:
{doc.page_content}
"""
        context_parts.append(context.strip())

    return "\n\n".join(context_parts)

In [11]:
user_query = "I want a 4 day Goa trip under 20000 with beaches, seafood and nightlife"

destination_result = {
    "agent_name": "Destination Agent",
    "status": "success",
    "recommended_destination": "Goa",
    "reason": "Goa matches beaches, seafood, nightlife and relaxed coastal travel.",
    "suitable_for": ["beach lovers", "food lovers", "nightlife travelers"],
    "suggested_duration": "3 to 5 days",
    "best_time_to_visit": "November to February",
    "confidence": "high"
}

context = get_itinerary_context(user_query, destination_result)

print(context)

Context 1
City: goa
Source: goa.txt
Content:
Best Time to Visit:
November to February is the most comfortable and popular season. Monsoon is scenic but beach activities may be limited.

Ideal Duration:
3 to 5 days

Travel Tips:
- Rent a scooter for local travel if comfortable.
- North Goa is better for nightlife.
- South Goa is better for peaceful beaches and luxury stays.

Context 2
City: goa
Source: goa.txt
Content:
Destination: Goa
State/Region: Goa
Destination Type: Beach, nightlife, seafood, coastal vacation, friends trip, couples trip

Overview:
Goa is one of India's most popular beach destinations. It is known for beaches, nightlife, seafood, Portuguese heritage, forts, churches, markets, and relaxed coastal culture.

Best For:
- Beach lovers
- Seafood lovers
- Friends groups
- Couples
- Nightlife travelers
- Relaxed vacation seekers

Context 3
City: goa
Source: goa.txt
Content:
Top Attractions:
- Baga Beach
- Calangute Beach
- Anjuna Beach
- Vagator Beach
- Fort Aguada
- Chapor

In [12]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3
)

print("Groq LLM loaded successfully.")

Groq LLM loaded successfully.


In [13]:
itinerary_agent_prompt = ChatPromptTemplate.from_template("""
You are the Itinerary Agent of VoyageAI.

Your task is to create the final travel itinerary by combining:
1. User travel request
2. Destination Agent output
3. Hotel Agent output
4. Food Agent output
5. Transport Agent output
6. Weather Agent output
7. Budget Agent output
8. Retrieved destination knowledge

User Travel Request:
{user_query}

Destination Agent Output:
{destination_result}

Hotel Agent Output:
{hotel_result}

Food Agent Output:
{food_result}

Transport Agent Output:
{transport_result}

Weather Agent Output:
{weather_result}

Budget Agent Output:
{budget_result}

Retrieved Destination Knowledge:
{context}

Return your answer strictly in valid JSON format.

JSON structure:
{{
  "agent_name": "Itinerary Agent",
  "status": "success/needs_budget_revision/skipped",
  "destination": "destination name",
  "trip_duration_days": number,
  "budget_status": "within_budget/over_budget/budget_not_provided/unknown",
  "summary": "short trip summary",
  "day_wise_itinerary": [
    {{
      "day": 1,
      "theme": "day theme",
      "morning": "morning plan",
      "afternoon": "afternoon plan",
      "evening": "evening plan",
      "food_suggestion": "food idea for the day",
      "transport_note": "local/intercity transport note"
    }}
  ],
  "stay_summary": "hotel or stay recommendation summary",
  "food_summary": "food recommendation summary",
  "transport_summary": "transport recommendation summary",
  "weather_summary": "seasonal weather advice summary",
  "budget_summary": {{
    "user_budget": number or null,
    "estimated_cost": number or null,
    "remaining_budget": number or null,
    "budget_status": "within_budget/over_budget/budget_not_provided/unknown"
  }},
  "budget_revision_notes": ["note 1", "note 2"],
  "travel_tips": ["tip 1", "tip 2", "tip 3"],
  "limitations": "short limitation statement",
  "confidence": "high/medium/low"
}}

Rules:
- Use only the retrieved destination knowledge and agent outputs.
- Do not invent exact hotel names.
- Do not invent live prices.
- Do not invent live weather.
- Do not invent train names, flight numbers, or bus operators.
- If Budget Agent says over_budget, set status to "needs_budget_revision".
- If Budget Agent says over_budget, include budget_revision_notes.
- If Budget Agent says within_budget, set status to "success".
- If budget is not provided, set budget_status to "budget_not_provided" and still generate itinerary.
- Keep the itinerary practical and beginner-friendly.
- Do not add markdown.
- Do not wrap JSON in ```json.
""")

In [15]:
itinerary_agent_chain = itinerary_agent_prompt | llm | StrOutputParser()

print("Itinerary Agent chain created successfully.")

Itinerary Agent chain created successfully.


In [17]:
def parse_json_response(response: str):
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        try:
            start = response.find("{")
            end = response.rfind("}") + 1
            json_str = response[start:end]
            return json.loads(json_str)
        except Exception:
            print("Failed to parse JSON. Raw response:")
            print(response)
            return {
                "error": "Failed to parse JSON",
                "raw_response": response
            }

In [18]:
def validate_required_agent_result(agent_result: dict, agent_name: str):
    if agent_result is None:
        return False, f"{agent_name} result is missing."

    status = agent_result.get("status")

    if status in ["skipped", "out_of_knowledge_base", "no_destination_found", "error"]:
        return False, f"{agent_name} did not return a valid result."

    return True, None

In [19]:
def run_itinerary_agent(
    user_query: str,
    destination_result: dict,
    hotel_result: dict = None,
    food_result: dict = None,
    transport_result: dict = None,
    weather_result: dict = None,
    budget_result: dict = None,
    k: int = 5
):
    destination_status = destination_result.get("status")
    destination = destination_result.get("recommended_destination")

    if destination_status != "success":
        return {
            "agent_name": "Itinerary Agent",
            "status": "skipped",
            "destination": None,
            "message": "Itinerary Agent skipped because Destination Agent did not return a valid destination.",
            "reason": destination_result.get("reason"),
            "confidence": "low"
        }

    if not destination:
        return {
            "agent_name": "Itinerary Agent",
            "status": "no_destination_found",
            "destination": None,
            "message": "Itinerary Agent cannot run because no destination was provided.",
            "confidence": "low"
        }

    if not is_destination_available(destination):
        return {
            "agent_name": "Itinerary Agent",
            "status": "out_of_knowledge_base",
            "destination": destination,
            "message": f"{destination} is not available in the current VoyageAI knowledge base.",
            "available_destinations": available_destinations,
            "confidence": "low"
        }

    context = get_itinerary_context(
        user_query=user_query,
        destination_result=destination_result,
        k=k
    )

    safe_hotel_result = hotel_result or {}
    safe_food_result = food_result or {}
    safe_transport_result = transport_result or {}
    safe_weather_result = weather_result or {}
    safe_budget_result = budget_result or {}

    response = itinerary_agent_chain.invoke({
        "user_query": user_query,
        "destination_result": json.dumps(destination_result, indent=2),
        "hotel_result": json.dumps(safe_hotel_result, indent=2),
        "food_result": json.dumps(safe_food_result, indent=2),
        "transport_result": json.dumps(safe_transport_result, indent=2),
        "weather_result": json.dumps(safe_weather_result, indent=2),
        "budget_result": json.dumps(safe_budget_result, indent=2),
        "context": context
    })

    parsed_response = parse_json_response(response)

    budget_status = safe_budget_result.get("budget_status", "unknown")

    if budget_status == "over_budget":
        parsed_response["status"] = "needs_budget_revision"
    elif budget_status == "within_budget":
        parsed_response["status"] = parsed_response.get("status", "success")
    elif budget_status == "budget_not_provided":
        parsed_response["status"] = parsed_response.get("status", "success")

    parsed_response["budget_status"] = budget_status

    return parsed_response

In [22]:
user_query = "I want a 5 day Goa trip from Bhubaneswar under 20000 with beaches, seafood and nightlife. Keep it low budget."

destination_result = {
    "agent_name": "Destination Agent",
    "status": "success",
    "recommended_destination": "Goa",
    "reason": "Goa matches beaches, seafood, nightlife and relaxed coastal travel.",
    "suitable_for": ["beach lovers", "food lovers", "nightlife travelers"],
    "suggested_duration": "3 to 5 days",
    "best_time_to_visit": "November to February",
    "confidence": "high"
}

hotel_result = {
    "agent_name": "Hotel Agent",
    "status": "success",
    "destination": "Goa",
    "budget_preference": "budget",
    "recommended_stay_type": "Hostel or budget stay near Anjuna, Vagator, or Baga",
    "best_areas": ["Anjuna", "Vagator", "Baga"],
    "stay_options": {
        "budget": "Hostels near Anjuna, Vagator, or Baga",
        "comfort": "Boutique hotels near Candolim or Panjim",
        "luxury": "Beach resorts in South Goa"
    },
    "confidence": "high"
}

food_result = {
    "agent_name": "Food Agent",
    "status": "success",
    "destination": "Goa",
    "food_preference_detected": "seafood",
    "recommended_foods": [
        "Goan fish curry",
        "Prawn balchao",
        "Pork vindaloo",
        "Xacuti",
        "Bebinca",
        "Seafood thali"
    ],
    "confidence": "high"
}

transport_result = {
    "agent_name": "Transport Agent",
    "status": "success",
    "source_city": "Bhubaneswar",
    "destination": "Goa",
    "travel_style_detected": "budget",
    "recommended_intercity_mode": "train",
    "local_transport_options": ["scooter rental", "local taxi", "walking near beach areas"],
    "confidence": "medium"
}

weather_result = {
    "agent_name": "Weather Agent",
    "status": "success",
    "destination": "Goa",
    "best_time_to_visit": "November to February",
    "season_suitability": "high",
    "weather_summary": "January is suitable for beaches, seafood and nightlife.",
    "confidence": "high"
}

budget_result = {
    "agent_name": "Budget Agent",
    "status": "success",
    "destination": "Goa",
    "trip_duration_days": 5,
    "estimated_nights": 3,
    "user_budget": 20000,
    "currency": "INR",
    "budget_level_detected": "budget",
    "transport_mode_used": "train",
    "estimated_cost_breakdown": {
        "transport": 3000,
        "hotel": 3960,
        "food": 3080,
        "activities": 2640,
        "local_transport": 1760,
        "buffer": 1444
    },
    "total_estimated_cost": 15884,
    "remaining_budget": 4116,
    "budget_status": "within_budget",
    "confidence": "medium"
}

itinerary_result = run_itinerary_agent(
    user_query=user_query,
    destination_result=destination_result,
    hotel_result=hotel_result,
    food_result=food_result,
    transport_result=transport_result,
    weather_result=weather_result,
    budget_result=budget_result
)

print(json.dumps(itinerary_result, indent=2))

{
  "agent_name": "Itinerary Agent",
  "status": "success",
  "destination": "Goa",
  "trip_duration_days": 5,
  "budget_status": "within_budget",
  "summary": "A 5-day Goa trip from Bhubaneswar with beaches, seafood, and nightlife, staying in budget-friendly hostels near Anjuna, Vagator, or Baga.",
  "day_wise_itinerary": [
    {
      "day": 1,
      "theme": "Arrival and Beach Time",
      "morning": "Arrive in Goa by train from Bhubaneswar. Check-in to a budget-friendly hostel near Anjuna, Vagator, or Baga.",
      "afternoon": "Visit Baga Beach and enjoy water sports or simply relax on the beach.",
      "evening": "Explore the nightlife of Baga and enjoy seafood at a local restaurant.",
      "food_suggestion": "Try Goan fish curry or Prawn balchao for dinner.",
      "transport_note": "Local taxi or walking near beach areas"
    },
    {
      "day": 2,
      "theme": "Beach Hopping",
      "morning": "Visit Anjuna Beach and enjoy the laid-back atmosphere.",
      "afternoon": "